# Comparativa de modelos - Clasificación

Objetivo: comparar dos versiones de cada modelo de clasificación (excepto la red bayesiana) usando validación cruzada y un conjunto holdout sobre Titanic.

## 1) Librerías

In [ ]:
!pip install pandas matplotlib seaborn scikit-learn


In [ ]:
# Importamos las bibliotecas principales del flujo de trabajo.
# La comparación completa se apoya en sklearn para preprocesamiento, validación cruzada y métricas.
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import get_scorer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier


## 2) Cargar datos

In [ ]:
# Leemos el dataset y revisamos su estructura antes de modelar.
data = pd.read_csv("titanic.csv")
data.head()


In [ ]:
# info() permite ver tipos de datos, faltantes y tamaño del dataset.
data.info()


## 3) Preparar datos

In [ ]:
# Definimos explícitamente qué columnas entran al modelo.
# Mantener esta lista visible ayuda a que el alumno entienda qué decisiones de diseño estamos tomando.
variables_a_descartar = ["PassengerId", "Name", "Cabin", "Ticket"]
variables_categoricas = ["Sex", "Embarked"]
variables_numericas = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
variable_dependiente = "Survived"

X = data.drop([variable_dependiente] + variables_a_descartar, axis=1)
y = data[variable_dependiente].astype(int)

print(X.head())
print(y.head())


## 4) Definir holdout y métricas

In [ ]:
# Primero reservamos un holdout.
# La validación cruzada se hará SOLO dentro del conjunto de entrenamiento.
random_state = 42
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=random_state, stratify=y
)

# El alumno puede cambiar estas dos métricas desde la lista disponible.
metricas_disponibles = ["accuracy", "precision", "recall", "f1", "roc_auc"]
metricas_seleccionadas = ["accuracy", "roc_auc"]

for metrica in metricas_seleccionadas:
    if metrica not in metricas_disponibles:
        raise ValueError(f"La métrica {metrica} no está en la lista disponible")

scoring = {metrica: metrica for metrica in metricas_seleccionadas}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)


## 5) Grilla de comparación

In [ ]:
# Definimos un preprocesamiento común para todos los modelos.
# Esto imita la lógica que ya usamos en los notebooks de clasificación.
preprocesamiento = ColumnTransformer([
    (
        "numerico",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        variables_numericas,
    ),
    (
        "categorico",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        variables_categoricas,
    ),
])

# La grilla no busca todos los hiperparámetros posibles.
# Aquí comparamos dos versiones razonables de cada modelo para discutir diferencias de desempeño.
comparison_grid = [
    {"modelo": "KNN", "version": "k=5", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", KNeighborsClassifier(n_neighbors=5))])},
    {"modelo": "KNN", "version": "k=11", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", KNeighborsClassifier(n_neighbors=11))])},
    {"modelo": "Regresión logística", "version": "C=0.5", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", LogisticRegression(C=0.5, max_iter=1000, random_state=random_state))])},
    {"modelo": "Regresión logística", "version": "C=2.0", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", LogisticRegression(C=2.0, max_iter=1000, random_state=random_state))])},
    {"modelo": "Árbol de decisión", "version": "max_depth=4", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", DecisionTreeClassifier(max_depth=4, random_state=random_state))])},
    {"modelo": "Árbol de decisión", "version": "max_depth=8", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", DecisionTreeClassifier(max_depth=8, random_state=random_state))])},
    {"modelo": "Random forest", "version": "200 árboles", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", RandomForestClassifier(n_estimators=200, random_state=random_state))])},
    {"modelo": "Random forest", "version": "400 árboles", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", RandomForestClassifier(n_estimators=400, random_state=random_state))])},
    {"modelo": "SVM", "version": "C=1", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", SVC(C=1.0, kernel="rbf", probability=True, random_state=random_state))])},
    {"modelo": "SVM", "version": "C=10", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", SVC(C=10.0, kernel="rbf", probability=True, random_state=random_state))])},
    {"modelo": "Naive Bayes", "version": "smoothing=1e-09", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", GaussianNB(var_smoothing=1e-9))])},
    {"modelo": "Naive Bayes", "version": "smoothing=1e-07", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", GaussianNB(var_smoothing=1e-7))])},
    {"modelo": "Red neuronal", "version": "(32,)", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", MLPClassifier(hidden_layer_sizes=(32,), max_iter=1000, random_state=random_state))])},
    {"modelo": "Red neuronal", "version": "(64, 32)", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=random_state))])},
]

pd.DataFrame([{"modelo": item["modelo"], "version": item["version"]} for item in comparison_grid])


## 6) CV out-of-set + holdout

In [ ]:
# cross_validate calcula métricas fuera de la muestra dentro de cada fold.
# Luego entrenamos el mismo pipeline sobre todo el training set y medimos el holdout.
resultados = []

for item in comparison_grid:
    pipeline = item["pipeline"]

    cv_scores = cross_validate(
        estimator=pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=None,
    )

    pipeline.fit(X_train, y_train)

    fila = {
        "modelo": item["modelo"],
        "version": item["version"],
    }

    for metrica in metricas_seleccionadas:
        cv_mean = cv_scores[f"test_{metrica}"].mean()
        cv_std = cv_scores[f"test_{metrica}"].std()
        holdout_score = get_scorer(metrica)(pipeline, X_holdout, y_holdout)

        fila[f"cv_mean_{metrica}"] = cv_mean
        fila[f"cv_std_{metrica}"] = cv_std
        fila[f"holdout_{metrica}"] = holdout_score
        fila[f"gap_{metrica}"] = holdout_score - cv_mean

    resultados.append(fila)

resultados_df = pd.DataFrame(resultados)
resultados_df


## 7) Comparar resultados

In [ ]:
# Ordenamos por la primera métrica elegida para facilitar la lectura.
metrica_principal = metricas_seleccionadas[0]
columnas = ["modelo", "version"]
for metrica in metricas_seleccionadas:
    columnas.extend([f"cv_mean_{metrica}", f"cv_std_{metrica}", f"holdout_{metrica}", f"gap_{metrica}"])

comparacion = resultados_df[columnas].sort_values(by=f"cv_mean_{metrica_principal}", ascending=False)
comparacion


In [ ]:
# Una visualización simple ayuda a ver si el rendimiento del holdout sigue la misma historia que la CV.
plt.figure(figsize=(12, 6))
sns.barplot(data=comparacion, x="modelo", y=f"cv_mean_{metrica_principal}", hue="version")
plt.title(f"Comparación por CV según {metrica_principal}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
